# Metronome Compass Testing

This notebook provides utilities for testing the complex `metronome_compass` (full battle tracking) by generating seeds that produce specific outcomes.

In [1]:
%load_ext autoreload
%autoreload 2
from claytonlib.safari import advance_rng
from claytonlib.metronome_compass import precompute_path, render_path
from claytonlib.moves import _moves_by_number, resolve_move
import datetime as dt

## RNG Reversing Utility

LCRNG: $state_{n+1} = (state_n \cdot 1103515245 + 24691) \pmod{2^{32}}$

Inverse: $state_n = ((state_{n+1} - 24691) \cdot 0xEEB9EB65) \pmod{2^{32}}$

In [2]:
def reverse_rng(state: int, n: int = 1) -> int:
    """Backtrack the LCRNG n steps."""
    inv_mult = 0xEEB9EB65
    for _ in range(n):
        state = ((state - 24691) * inv_mult) & 0xFFFFFFFF
    return state


def generate_seed_for_move(move_name: str, magikarp_level: int = 2) -> dict:
    """Find seeds producing a specific Metronome move on Turn 1, one per Magikarp scenario.

    Turn order: Magikarp moves first, then Metronome user.
    Advances before the Metronome roll (advance #N → seed = reverse_rng(target, N)):
      Splash (any level):  6+1+4+0+2+2 = 15  → N=16  (check rev[9]  even  → Splash)
      Tackle miss:         6+1+4+3+0+2 = 16  → N=17  (check rev[10] odd + rev[3]  >=95 → miss)
      Tackle hit:          6+1+4+3+2+2 = 18  → N=19  (check rev[12] odd + rev[5]  <95  → hit)

    Tries up to 200 multiples of the pool (467) in the top-16 bits to satisfy each scenario.
    Returns dict mapping scenario name → seed.
    """
    move = resolve_move(move_name)
    if not move or not move.metronome_usable:
        raise ValueError(f"Move {move_name!r} is not metronome-usable.")

    target_num = move.number
    pool = 467
    target_val = target_num - 1
    results = {}

    for attempt in range(200):
        top16 = (target_val + pool * attempt) % 65536
        target_state = top16 << 16

        # Build rev[1..19]: rev[n] = reverse_rng(target_state, n)
        rev = [None]
        s = target_state
        for _ in range(19):
            s = ((s - 24691) * 0xEEB9EB65) & 0xFFFFFFFF
            rev.append(s)

        if magikarp_level < 15:
            if 'splash' not in results:
                results['splash'] = rev[16]
        else:
            # Move-select roll is advance #7 from seed; hit roll is advance #14.
            # For seed=rev[N]: advance #k from seed = rev[N-k].
            if 'splash' not in results and (rev[9] >> 16) % 2 == 0:
                results['splash'] = rev[16]
            if 'tackle_hit' not in results and (rev[12] >> 16) % 2 == 1 and (rev[5] >> 16) % 100 < 95:
                results['tackle_hit'] = rev[19]
            if 'tackle_miss' not in results and (rev[10] >> 16) % 2 == 1 and (rev[3] >> 16) % 100 >= 95:
                results['tackle_miss'] = rev[17]

        target_count = 1 if magikarp_level < 15 else 3
        if len(results) == target_count:
            break

    return results


def verify_seed(seed: int, magikarp_level: int = 2):
    path = precompute_path(seed, magikarp_level=magikarp_level, opposite_gender=False, n_turns=3)
    print(f"Seed: 0x{seed:08X}")
    print(f"Path: {render_path(path)}")
    moves = _moves_by_number()
    if path:
        for token in path[0]:
            if hasattr(token, 'move_num'):
                print(f"Move: {moves[token.move_num].name} (M{token.move_num:03d})")
                break

## Example: Seed for Flamethrower

In [ ]:
seeds = generate_seed_for_move("Flamethrower")
verify_seed(seeds['splash'])

## Example: Seed for Splash (Turn 1 Metronome)

In [ ]:
seeds = generate_seed_for_move("Splash")
verify_seed(seeds['splash'])

## Testing other moves

In [ ]:
seeds = generate_seed_for_move("Taunt")
verify_seed(seeds['splash'])

## Multi-Turn Path Generation

In [ ]:
seed = 0x12345678
path = precompute_path(seed, magikarp_level=15, opposite_gender=False, n_turns=5)
print(f"Seed: 0x{seed:08X} (Level 15 Magikarp)")
print(f"Path: {render_path(path)}")

## All 3 Magikarp Scenarios (Level 15)

In [4]:
seeds = generate_seed_for_move("Flamethrower", magikarp_level=15)
moves = _moves_by_number()

for scenario, seed in seeds.items():
    print(f"=== {scenario} ===")
    verify_seed(seed, magikarp_level=15)
    print()

=== tackle_hit ===
Seed: 0x9836B0CF
Path: KtkhM053h KspM400h KtkhM238h
Move: Flamethrower (M053)

=== splash ===
Seed: 0x871A3EF0
Path: KspM053h Ktk-M051h KspM414h
Move: Flamethrower (M053)

=== tackle_miss ===
Seed: 0x21C48651
Path: Ktk-M053h~ KtkhM423h KtkhM273
Move: Flamethrower (M053)

